# 10 Customer analysis from the shared dataset

**Questions:** Who purchases, how many people have more than one observed order, and how often do mature cohorts buy again on a later date?

Run 04 first. This notebook reads canonical tables and applies explicit analysis populations; it does not clean the raw data again. The customer population is the dataset's `customer_analysis_eligible` flag: positive website or Shopee purchases, excluding configured special accounts. Other platforms remain in sales totals, but their person-level repeat behaviour is not established by the supplier mapping.

Only aggregate summaries are displayed. A person's first observed purchase is not necessarily their lifetime first purchase. Repeat rates are associations within the observation window, not intervention effects.


In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))  # run from the notebooks folder

import pandas as pd
from IPython.display import display
from weavve.clean import load_dataset

dataset = load_dataset()
print("Shared dataset ID:", dataset.manifest["dataset_id"])
print("Local dataset:", dataset.path)
print("Transaction cutoff:", dataset.manifest["transaction_cutoff"])
print("Review flags:", len(dataset.manifest["warnings"]))
pd.set_option("display.max_columns", 20)


## Reconcile sales and customer populations

The channel table keeps every noncancelled order, including zero or negative signed net sales after refunds. Purchase counts use `is_purchase`; the customer-analysis count uses the stricter shared eligibility flag. This makes the different denominators visible.


In [ ]:
orders = dataset.frame("orders")
channel_summary = orders.groupby("channel_detail", dropna=False).agg(
    orders=("order_id", "size"),
    purchases=("is_purchase", "sum"),
    customer_analysis_orders=("customer_analysis_eligible", "sum"),
    special_account_orders=("is_special_account", "sum"),
    signed_net_sales=("net", "sum"),
)
channel_summary["signed_net_sales"] = channel_summary["signed_net_sales"].map(float).round(2)
display(channel_summary)

eligible = orders.loc[orders["customer_analysis_eligible"]].copy()
people = eligible.groupby("person_id").agg(orders=("order_id", "size"), signed_net=("net", "sum"))
display(pd.DataFrame([{
    "eligible_purchases": len(eligible),
    "observed_people": len(people),
    "people_with_two_or_more_orders": int((people["orders"] >= 2).sum()),
    "two_or_more_orders_pct": 100 * (people["orders"] >= 2).mean() if len(people) else None,
}]))


## Mature 90-day cohorts

Each person enters on their earliest eligible observed purchase, with order ID breaking same-date ties. A second purchase must occur on a strictly later calendar date and no more than 90 days later. Only people with all 90 days available before the transaction cutoff enter the denominator. Cohorts are grouped by the calendar year and channel of that first observed purchase.

This is a website/Shopee purchasing view. It excludes same-day additional orders and does not identify whether a purchase was caused by an email or loyalty reward.


In [ ]:
from datetime import date, timedelta

cutoff = date.fromisoformat(dataset.manifest["transaction_cutoff"])
first = eligible.sort_values(["order_date", "order_id"]).drop_duplicates("person_id").set_index("person_id")
mature = first.loc[first["order_date"].map(lambda d: d + timedelta(days=90) <= cutoff)].copy()
followup = eligible.join(mature[["order_date"]].rename(columns={"order_date": "first_observed_date"}), on="person_id", how="inner")
followup["days_after_first"] = (pd.to_datetime(followup["order_date"]) - pd.to_datetime(followup["first_observed_date"])).dt.days
returned = set(followup.loc[followup["days_after_first"].between(1, 90), "person_id"])
mature["later_purchase_within_90d"] = mature.index.isin(returned)
mature["first_observed_year"] = mature["order_date"].map(lambda d: d.year)
cohorts = mature.groupby(["first_observed_year", "channel_detail"]).agg(
    eligible_people=("order_id", "size"), later_purchase_people=("later_purchase_within_90d", "sum"),
)
cohorts["later_purchase_pct"] = 100 * cohorts["later_purchase_people"] / cohorts["eligible_people"]
display(cohorts.round(2))


## Voucher snapshot coverage

File F contains issued vouchers and their recorded usage at the source snapshot. It is not a complete membership or points ledger. A link to File G establishes an order identifier match; it does not by itself establish a completed positive purchase. No voucher status below is interpreted as a usage event observed by the transaction cutoff.


In [ ]:
rewards = dataset.frame("loyalty_rewards")
voucher_summary = rewards.groupby("clean_reward_usage_status", dropna=False).agg(
    issued_voucher_rows=("source_row", "size"), linked_to_source_order=("clean_order_in_g", "sum"),
)
display(voucher_summary)
